# Generate Free Shipping Experiment Data

This notebook regenerates the synthetic dataset used in the free shipping threshold experiment.

It performs the following steps:

1. Sets project paths
2. Loads the experiment configuration
3. Imports the synthetic data generator
4. Generates the dataset
5. Saves the dataset to `data/synthetic/`
6. Runs basic sanity checks

In [2]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import yaml

In [3]:


PROJECT_ROOT = Path.cwd()

CONFIG_PATH = PROJECT_ROOT / "configs" / "experiment_config.yaml"
OUTPUT_DIR = PROJECT_ROOT / "data" / "synthetic"
OUTPUT_PATH = PROJECT_ROOT / "data" / "synthetic" / "free_shipping_experiment_sessions.csv"

print(CONFIG_PATH.exists())

True


In [4]:
from src.generate_free_shipping_experiment import (
    load_config,
    generate_calendar,
    estimate_visitor_pool_size,
    generate_visitors,
    generate_sessions,
    assign_experiment_arms,
    simulate_conversion,
    simulate_orders_and_financials,
    summarize_experiment,
    run_diagnostics,
)

In [5]:
config = load_config(CONFIG_PATH)

config

{'experiment': {'arms': {'t35': {'conversion_rate': 0.0225,
    'target_aov': 44,
    'free_shipping_qualification_rate': 0.7},
   't50': {'conversion_rate': 0.02,
    'target_aov': 55,
    'free_shipping_qualification_rate': 0.5},
   't65': {'conversion_rate': 0.017,
    'target_aov': 59,
    'free_shipping_qualification_rate': 0.33}}},
 'economics': {'shipping_cost_per_order': 8.99,
  'shipping_fee_if_not_qualified': 5.99,
  'cogs_ratio': 0.6},
 'traffic': {'avg_sessions_per_day': 10000, 'session_std_dev': 800},
 'experiment_dates': {'pre_period_start': '2025-01-01',
  'test_start': '2025-06-01',
  'test_end': '2025-07-15'},
 'basket_model': {'avg_units_per_order': 2.5,
  'units_std_dev': 1.0,
  'unit_price_mean': 20,
  'unit_price_std_dev': 5,
  'min_unit_price': 5,
  'max_unit_price': 80}}

In [6]:
SEED = 42
rng = np.random.default_rng(SEED)

# 1. calendar
calendar_df = generate_calendar(config)

# 2. estimate visitor pool
n_visitors = estimate_visitor_pool_size(
    calendar_df=calendar_df,
    avg_sessions_per_day=int(config["traffic"]["avg_sessions_per_day"]),
    avg_sessions_per_visitor=3.0,
)

# 3. visitors
visitors_df = generate_visitors(
    n_visitors=n_visitors,
    config=config,
    rng=rng,
)

# 4. sessions
sessions_df = generate_sessions(
    calendar_df=calendar_df,
    visitors_df=visitors_df,
    config=config,
    rng=rng,
)

# 5. experiment arm assignment
assigned_df = assign_experiment_arms(
    sessions_df=sessions_df,
    visitors_df=visitors_df,
)

# 6. conversions
converted_df = simulate_conversion(
    df=assigned_df,
    config=config,
    rng=rng,
)

# 7. orders + economics
final_df = simulate_orders_and_financials(
    df=converted_df,
    config=config,
    rng=rng,
)

print("Final dataset shape:", final_df.shape)
final_df.head()

Final dataset shape: (1959144, 23)


,session_id,visitor_id,session_date,period,region,test_arm,conversion_multiplier,basket_multiplier,threshold_arm,free_shipping_threshold,...,order_id,units,price_per_unit,product_revenue,shipping_revenue,total_revenue,cogs,shipping_cost,contribution_margin,qualified_for_free_shipping
0,1,650921,2025-01-01,pre,West,t65,1.306001,1.076006,pre,50.0,...,None,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,2,373732,2025-01-01,pre,West,t65,0.988317,1.140570,pre,50.0,...,None,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,3,338105,2025-01-01,pre,Central,t35,1.227563,0.882506,pre,50.0,...,None,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,4,156611,2025-01-01,pre,West,t35,1.123940,1.324733,pre,50.0,...,None,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,5,624327,2025-01-01,pre,West,t50,0.790399,0.764117,pre,50.0,...,None,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

final_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved dataset to: {OUTPUT_PATH}")

Saved dataset to: /Users/scottbelarmino/ds_decision_science_exp_repo/free_shipping_threshold_experiment/data/synthetic/free_shipping_experiment_sessions.csv


In [8]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1959144 entries, 0 to 1959143
Data columns (total 23 columns):
 #   Column                       Dtype         
---  ------                       -----         
 0   session_id                   int64         
 1   visitor_id                   int64         
 2   session_date                 datetime64[us]
 3   period                       str           
 4   region                       str           
 5   test_arm                     str           
 6   conversion_multiplier        float64       
 7   basket_multiplier            float64       
 8   threshold_arm                str           
 9   free_shipping_threshold      float64       
 10  base_conversion_rate         float64       
 11  conversion_probability       float64       
 12  converted                    int64         
 13  order_id                     object        
 14  units                        int64         
 15  price_per_unit               float64       
 16  product_rev

In [9]:
print("Date range:")
print(final_df["session_date"].min(), "to", final_df["session_date"].max())

print("\nRows by period:")
display(final_df.groupby("period").size().rename("rows").reset_index())

Date range:
2025-01-01 00:00:00 to 2025-07-15 00:00:00

Rows by period:


,period,rows
0,pre,1509602
1,test,449542


In [10]:
test_df = final_df[final_df["period"] == "test"].copy()

sessions_by_arm = (
    test_df.groupby("threshold_arm")
    .size()
    .rename("sessions")
    .reset_index()
    .sort_values("threshold_arm")
)

sessions_by_arm

,threshold_arm,sessions
0,t35,149618
1,t50,150306
2,t65,149618


In [11]:
summary = summarize_experiment(final_df)
summary

,threshold_arm,sessions,orders,conversion_rate,aov,revenue_per_session,cm_per_session,free_shipping_qualification_rate,shipping_subsidy_per_order,negative_margin_order_rate
0,t35,149618,3306,0.022096,44.436044,1.019744,0.231978,0.713854,7.275983,0.0
1,t50,150306,3075,0.020458,55.404153,1.195602,0.331599,0.493008,5.953119,0.0
2,t65,149618,2623,0.017531,59.992493,1.122329,0.333675,0.327869,4.963934,0.0


In [13]:
summary_output_path = OUTPUT_DIR / "free_shipping_experiment_summary.csv"
summary.to_csv(summary_output_path, index=False)

print(f"Saved summary to: {summary_output_path}")

Saved summary to: /Users/scottbelarmino/ds_decision_science_exp_repo/free_shipping_threshold_experiment/data/synthetic/free_shipping_experiment_summary.csv


## Output

This notebook generated:

- `data/synthetic/free_shipping_experiment_sessions.csv`
- `data/synthetic/free_shipping_experiment_summary.csv`

The next step is to open:

`notebooks/01_experiment_analysis.ipynb`

to reproduce the full experiment review.